# llmon on Google Colab — one click

Run the two cells below in order. Cell 1 installs Rust + builds `llmon` (~1–2 min, one time). Cell 2 pulls a small GGUF and chats with it through the OpenAI-compatible API.

> Colab free tier has no GPU guarantee; the default `Qwen2.5-0.5B Q4_K_M` (~400 MB) runs fine on CPU.

In [ ]:
%%bash
set -e
if ! command -v cargo >/dev/null; then curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal; fi
export PATH="$HOME/.cargo/bin:$PATH"
test -d /content/llmon || git clone --depth 1 https://github.com/deepsuthar496/llmon /content/llmon
cd /content/llmon && cargo build --release --locked
export LLMON_PORT=11435
pkill -f 'llmon serve' || true
nohup /content/llmon/target/release/llmon serve >/tmp/llmon.log 2>&1 &
sleep 3 && curl -s localhost:11435/health && echo && /content/llmon/target/release/llmon bench --tokens 64

In [ ]:
%%bash
set -e
BIN=/content/llmon/target/release/llmon
$BIN pull Qwen/Qwen2.5-0.5B-Instruct-GGUF/qwen2.5-0.5b-instruct-q4_k_m.gguf
$BIN list
curl -s -H 'Content-Type: application/json' localhost:11435/api/generate \
  -d '{"model":"Qwen","prompt":"Hello from Colab!","stream":false}'

### Use it like OpenAI

```python
import json, urllib.request
req = urllib.request.Request(
    "http://localhost:11435/v1/chat/completions",
    data=json.dumps({"model": "Qwen", "messages": [{"role": "user", "content": "hi"}]}).encode(),
    headers={"Content-Type": "application/json"},
)
print(json.load(urllib.request.urlopen(req))["choices"][0]["message"]["content"])
```